In [5]:
pip install paddleocr

Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import filedialog, ttk, scrolledtext
import os
from tkinter import messagebox
import threading
import time
import sys
import subprocess

class PrescriptionOCRApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Prescription OCR System")
        self.root.geometry("1200x800")
        
        # --- Model Configuration ---
        self.char_list = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        self.img_size = (32, 128)  # Height, Width expected by CRNN model
        
        # Variables
        self.image_path = None
        self.original_image = None
        self.displayed_image = None
        self.tk_image = None
        self.crop_start_x = None
        self.crop_start_y = None
        self.crop_rect = None
        self.is_cropping = False
        self.cropped_image = None
        self.model_loaded = False
        self.ocr_detector = None
        self.crnn_model = None
        self.dependency_check_complete = False
        self.zoom_factor = 1.0
        self.image_enhancements = {
            "brightness": 0,
            "contrast": 1.0,
            "sharpness": 1.0
        }
        
        # Create UI
        self.create_ui()
        
        # Check dependencies in a separate thread
        self.check_thread = threading.Thread(target=self.check_and_install_dependencies)
        self.check_thread.daemon = True
        self.check_thread.start()
    
    def check_and_install_dependencies(self):
        """Check if required packages are installed and install them if needed"""
        self.status_var.set("Checking dependencies...")
        
        # Check for paddleocr
        try:
            import importlib.util
            paddle_spec = importlib.util.find_spec("paddleocr")
            paddle_installed = paddle_spec is not None
        except ImportError:
            paddle_installed = False
        
        if not paddle_installed:
            self.status_var.set("Installing PaddleOCR (this may take a few minutes)...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "paddlepaddle", "paddleocr"])
                self.status_var.set("PaddleOCR installed successfully")
                paddle_installed = True
            except subprocess.CalledProcessError:
                self.status_var.set("Failed to install PaddleOCR. Please install manually: pip install paddlepaddle paddleocr")
        
        # Now that dependencies are installed (or at least checked), import paddleocr
        if paddle_installed:
            try:
                from paddleocr import PaddleOCR
                self.status_var.set("Loading OCR model...")
                self.ocr_detector = PaddleOCR(use_angle_cls=True, lang='en')
                self.status_var.set("OCR model loaded successfully")
            except Exception as e:
                self.status_var.set(f"Error loading OCR model: {e}")
        
        # Load CRNN model if path is provided
        model_path = os.path.join( "C:\Data Set\Trained model\tanmay model_Gem.keras")  # Fixed path with os.path.join
        if os.path.exists(model_path):
            try:
                self.status_var.set("Loading CRNN model...")
                from tensorflow.keras.models import load_model
                self.crnn_model = load_model(model_path)
                self.model_loaded = True
                self.status_var.set("CRNN model loaded successfully")
            except Exception as e:
                self.status_var.set(f"Error loading CRNN model: {e}")
                self.model_loaded = False
        else:
            self.status_var.set(f"CRNN model not found at {model_path}")
            self.model_loaded = False
        
        self.dependency_check_complete = True
        self.status_var.set("Ready")
    
    def create_ui(self):
        # Main frame with notebook for tabs
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Main OCR tab
        ocr_tab = ttk.Frame(self.notebook)
        self.notebook.add(ocr_tab, text="OCR")
        
        # Settings tab
        settings_tab = ttk.Frame(self.notebook)
        self.notebook.add(settings_tab, text="Settings")
        
        # Help tab
        help_tab = ttk.Frame(self.notebook)
        self.notebook.add(help_tab, text="Help")
        
        # Create OCR tab content
        self.create_ocr_tab(ocr_tab)
        
        # Create settings tab content
        self.create_settings_tab(settings_tab)
        
        # Create help tab content
        self.create_help_tab(help_tab)
        
        # Status bar
        status_frame = ttk.Frame(self.root)
        status_frame.pack(side=tk.BOTTOM, fill=tk.X)
        
        self.status_var = tk.StringVar()
        self.status_var.set("Starting up...")
        status_bar = ttk.Label(status_frame, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        status_bar.pack(side=tk.LEFT, fill=tk.X, expand=True)
        
        # Progress bar
        self.progress_var = tk.DoubleVar()
        self.progress = ttk.Progressbar(status_frame, variable=self.progress_var, length=100, mode='determinate')
        self.progress.pack(side=tk.RIGHT, padx=5)
    
    def create_ocr_tab(self, parent):
        # Main frame
        main_frame = ttk.Frame(parent, padding=10)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Left panel - Image display and controls
        left_panel = ttk.Frame(main_frame)
        left_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        # Control buttons
        control_frame = ttk.Frame(left_panel)
        control_frame.pack(fill=tk.X, pady=5)
        
        # First row of buttons
        ttk.Button(control_frame, text="Upload Image", command=self.upload_image).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame, text="Enable Crop", command=self.enable_crop_mode).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame, text="Reset Crop", command=self.reset_crop).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame, text="Predict", command=self.predict_threaded).pack(side=tk.LEFT, padx=5)
        
        # Second row of buttons
        control_frame2 = ttk.Frame(left_panel)
        control_frame2.pack(fill=tk.X, pady=5)
        
        ttk.Button(control_frame2, text="Zoom In", command=self.zoom_in).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame2, text="Zoom Out", command=self.zoom_out).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame2, text="Enhance Image", command=self.enhance_image).pack(side=tk.LEFT, padx=5)
        ttk.Button(control_frame2, text="Save Results", command=self.save_results).pack(side=tk.LEFT, padx=5)
        
        # Image canvas with scrollbars
        canvas_container = ttk.Frame(left_panel, borderwidth=2, relief="groove")
        canvas_container.pack(fill=tk.BOTH, expand=True, pady=5)
        
        # Add scrollbars
        h_scrollbar = ttk.Scrollbar(canvas_container, orient=tk.HORIZONTAL)
        h_scrollbar.pack(side=tk.BOTTOM, fill=tk.X)
        
        v_scrollbar = ttk.Scrollbar(canvas_container, orient=tk.VERTICAL)
        v_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Canvas for image display
        self.canvas = tk.Canvas(canvas_container, bg="lightgray", 
                               xscrollcommand=h_scrollbar.set,
                               yscrollcommand=v_scrollbar.set)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Configure scrollbars
        h_scrollbar.config(command=self.canvas.xview)
        v_scrollbar.config(command=self.canvas.yview)
        
        # Canvas events for cropping
        self.canvas.bind("<ButtonPress-1>", self.on_mouse_down)
        self.canvas.bind("<B1-Motion>", self.on_mouse_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_mouse_up)
        self.canvas.bind("<MouseWheel>", self.on_mouse_wheel)  # For Windows
        self.canvas.bind("<Button-4>", self.on_mouse_wheel)    # For Linux scroll up
        self.canvas.bind("<Button-5>", self.on_mouse_wheel)    # For Linux scroll down
        
        # Right panel - Results
        right_panel = ttk.Frame(main_frame, padding=(10, 0, 0, 0))
        right_panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Results area
        results_frame = ttk.LabelFrame(right_panel, text="Detected Text")
        results_frame.pack(fill=tk.BOTH, expand=True)
        
        self.result_text = scrolledtext.ScrolledText(results_frame, wrap=tk.WORD)
        self.result_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Confidence threshold slider
        threshold_frame = ttk.Frame(right_panel)
        threshold_frame.pack(fill=tk.X, pady=10)
        
        ttk.Label(threshold_frame, text="Confidence Threshold:").pack(side=tk.LEFT)
        
        self.threshold_var = tk.DoubleVar(value=0.5)
        threshold_slider = ttk.Scale(threshold_frame, from_=0.0, to=1.0, 
                                     variable=self.threshold_var, orient=tk.HORIZONTAL)
        threshold_slider.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        
        threshold_value = ttk.Label(threshold_frame, text="0.5")
        threshold_value.pack(side=tk.LEFT, padx=5)
        
        # Update threshold value label when slider changes
        def update_threshold_label(*args):
            threshold_value.config(text=f"{self.threshold_var.get():.2f}")
        
        self.threshold_var.trace_add("write", update_threshold_label)
    
    def create_settings_tab(self, parent):
        # Settings frame
        settings_frame = ttk.Frame(parent, padding=20)
        settings_frame.pack(fill=tk.BOTH, expand=True)
        
        # Model settings
        model_frame = ttk.LabelFrame(settings_frame, text="Model Settings", padding=10)
        model_frame.pack(fill=tk.X, pady=10)
        
        # Model path
        ttk.Label(model_frame, text="CRNN Model Path:").grid(row=0, column=0, sticky=tk.W, pady=5)
        
        self.model_path_var = tk.StringVar(value=os.path.join("C:\Data Set\Trained model\tanmay model_Gem.keras"))
        model_path_entry = ttk.Entry(model_frame, textvariable=self.model_path_var, width=50)
        model_path_entry.grid(row=0, column=1, sticky=tk.W, padx=5, pady=5)
        
        ttk.Button(model_frame, text="Browse", command=self.browse_model).grid(row=0, column=2, padx=5, pady=5)
        
        # Character list
        ttk.Label(model_frame, text="Character List:").grid(row=1, column=0, sticky=tk.W, pady=5)
        
        self.char_list_var = tk.StringVar(value=self.char_list)
        char_list_entry = ttk.Entry(model_frame, textvariable=self.char_list_var, width=50)
        char_list_entry.grid(row=1, column=1, columnspan=2, sticky=tk.W+tk.E, padx=5, pady=5)
        
        # Image enhancement settings
        enhancement_frame = ttk.LabelFrame(settings_frame, text="Image Enhancement", padding=10)
        enhancement_frame.pack(fill=tk.X, pady=10)
        
        # Brightness
        ttk.Label(enhancement_frame, text="Brightness:").grid(row=0, column=0, sticky=tk.W, pady=5)
        
        self.brightness_var = tk.IntVar(value=0)
        brightness_slider = ttk.Scale(enhancement_frame, from_=-50, to=50, 
                                     variable=self.brightness_var, orient=tk.HORIZONTAL)
        brightness_slider.grid(row=0, column=1, sticky=tk.W+tk.E, padx=5, pady=5)
        
        ttk.Label(enhancement_frame, textvariable=self.brightness_var).grid(row=0, column=2, padx=5)
        
        # Contrast
        ttk.Label(enhancement_frame, text="Contrast:").grid(row=1, column=0, sticky=tk.W, pady=5)
        
        self.contrast_var = tk.DoubleVar(value=1.0)
        contrast_slider = ttk.Scale(enhancement_frame, from_=0.5, to=2.0, 
                                   variable=self.contrast_var, orient=tk.HORIZONTAL)
        contrast_slider.grid(row=1, column=1, sticky=tk.W+tk.E, padx=5, pady=5)
        
        self.contrast_label = ttk.Label(enhancement_frame, text="1.0")
        self.contrast_label.grid(row=1, column=2, padx=5)
        
        # Sharpness
        ttk.Label(enhancement_frame, text="Sharpness:").grid(row=2, column=0, sticky=tk.W, pady=5)
        
        self.sharpness_var = tk.DoubleVar(value=1.0)
        sharpness_slider = ttk.Scale(enhancement_frame, from_=0.0, to=3.0, 
                                    variable=self.sharpness_var, orient=tk.HORIZONTAL)
        sharpness_slider.grid(row=2, column=1, sticky=tk.W+tk.E, padx=5, pady=5)
        
        self.sharpness_label = ttk.Label(enhancement_frame, text="1.0")
        self.sharpness_label.grid(row=2, column=2, padx=5)
        
        # Update labels when sliders change
        def update_enhancement_labels(*args):
            self.contrast_label.config(text=f"{self.contrast_var.get():.1f}")
            self.sharpness_label.config(text=f"{self.sharpness_var.get():.1f}")
            
            # Update image enhancements dictionary
            self.image_enhancements["brightness"] = self.brightness_var.get()
            self.image_enhancements["contrast"] = self.contrast_var.get()
            self.image_enhancements["sharpness"] = self.sharpness_var.get()
        
        self.brightness_var.trace_add("write", update_enhancement_labels)
        self.contrast_var.trace_add("write", update_enhancement_labels)
        self.sharpness_var.trace_add("write", update_enhancement_labels)
        
        # Apply and Save buttons
        button_frame = ttk.Frame(settings_frame)
        button_frame.pack(fill=tk.X, pady=20)
        
        ttk.Button(button_frame, text="Apply Settings", command=self.apply_settings).pack(side=tk.RIGHT, padx=5)
        ttk.Button(button_frame, text="Reset to Defaults", command=self.reset_settings).pack(side=tk.RIGHT, padx=5)
    
    def create_help_tab(self, parent):
        # Help frame
        help_frame = ttk.Frame(parent, padding=20)
        help_frame.pack(fill=tk.BOTH, expand=True)
        
        # Help text
        help_text = """
        # Prescription OCR System Help
        
        This application helps you extract text from prescription images using OCR technology.
        
        ## Basic Usage
        
        1. **Upload Image**: Click to select a prescription image file
        2. **Enable Crop**: Activate crop mode to select a specific region
        3. **Reset Crop**: Return to the full image view
        4. **Predict**: Run OCR on the current image
        5. **Zoom In/Out**: Adjust the image size for better viewing
        6. **Enhance Image**: Apply current enhancement settings
        7. **Save Results**: Export the detected text to a file
        
        ## Tips for Better Results
        
        - Ensure good lighting when capturing prescription images
        - Crop to focus on text areas only
        - Adjust enhancement settings for clearer text
        - Try different confidence thresholds for detection
        
        ## Troubleshooting
        
        If you encounter issues:
        - Check that your model path is correct in Settings
        - Ensure all dependencies are properly installed
        - Try restarting the application
        
        For more help, contact support.
        """
        
        help_text_widget = scrolledtext.ScrolledText(help_frame, wrap=tk.WORD)
        help_text_widget.pack(fill=tk.BOTH, expand=True)
        help_text_widget.insert(tk.END, help_text)
        help_text_widget.config(state=tk.DISABLED)  # Make read-only
    
    def browse_model(self):
        file_path = filedialog.askopenfilename(
            filetypes=[("Keras Model", "*.keras *.h5"), ("All Files", "*.*")]
        )
        
        if file_path:
            self.model_path_var.set(file_path)
    
    def apply_settings(self):
        # Update character list
        self.char_list = self.char_list_var.get()
        
        # Try to load model if path changed
        new_model_path = self.model_path_var.get()
        current_model_path = os.path.join("C:\Data Set\Trained model\tanmay model_Gem.keras")
        
        if new_model_path != current_model_path and os.path.exists(new_model_path):
            try:
                self.status_var.set("Loading new CRNN model...")
                from tensorflow.keras.models import load_model
                self.crnn_model = load_model(new_model_path)
                self.model_loaded = True
                self.status_var.set("CRNN model loaded successfully")
            except Exception as e:
                self.status_var.set(f"Error loading CRNN model: {e}")
                self.model_loaded = False
        
        # Apply image enhancements if an image is loaded
        if self.original_image is not None:
            self.enhance_image()
        
        messagebox.showinfo("Settings", "Settings applied successfully")
    
    def reset_settings(self):
        # Reset to default values
        self.char_list_var.set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
        self.model_path_var.set(os.path.join("C:\Data Set\Trained model\tanmay model_Gem.keras"))
        self.brightness_var.set(0)
        self.contrast_var.set(1.0)
        self.sharpness_var.set(1.0)
        self.threshold_var.set(0.5)
        
        # Reset image enhancements
        self.image_enhancements = {
            "brightness": 0,
            "contrast": 1.0,
            "sharpness": 1.0
        }
        
        # Apply to current image if loaded
        if self.original_image is not None:
            self.reset_crop()
    
    def upload_image(self):
        file_path = filedialog.askopenfilename(
            filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
        )
        
        if file_path:
            self.image_path = file_path
            self.status_var.set(f"Loaded image: {os.path.basename(file_path)}")
            self.load_image()
    
    def load_image(self):
        try:
            # Load image with OpenCV for processing
            self.original_image = cv2.imread(self.image_path)
            
            # Convert to RGB for display
            rgb_image = cv2.cvtColor(self.original_image, cv2.COLOR_BGR2RGB)
            self.displayed_image = rgb_image.copy()
            
            # Reset zoom
            self.zoom_factor = 1.0
            
            # Resize for display if needed
            self.resize_image_for_display()
            
            # Reset crop
            self.reset_crop()
            
            # Update status
            self.status_var.set(f"Image loaded: {os.path.basename(self.image_path)}")
        except Exception as e:
            self.status_var.set(f"Error loading image: {e}")
    
    def resize_image_for_display(self):
        if self.original_image is None:
            return
            
        # Get canvas size
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            # Canvas not yet realized, use default size
            canvas_width = 600
            canvas_height = 600
        
        # Get image dimensions
        img_height, img_width = self.displayed_image.shape[:2]
        
        # Apply zoom factor
        new_width = int(img_width * self.zoom_factor)
        new_height = int(img_height * self.zoom_factor)
        
        # Resize image
        resized_image = cv2.resize(self.displayed_image, (new_width, new_height))
        
        # Convert to PIL format for Tkinter
        pil_image = Image.fromarray(resized_image)
        self.tk_image = ImageTk.PhotoImage(image=pil_image)
        
        # Update canvas
        self.canvas.config(width=canvas_width, height=canvas_height)
        self.canvas.config(scrollregion=(0, 0, new_width, new_height))
        self.canvas.delete("all")
        self.canvas.create_image(0, 0, anchor=tk.NW, image=self.tk_image)
    
    def zoom_in(self):
        if self.original_image is not None:
            self.zoom_factor *= 1.2
            self.resize_image_for_display()
    
    def zoom_out(self):
        if self.original_image is not None:
            self.zoom_factor /= 1.2
            if self.zoom_factor < 0.1:
                self.zoom_factor = 0.1
            self.resize_image_for_display()
    
    def on_mouse_wheel(self, event):
        if self.original_image is not None:
            # Determine zoom direction
            if event.num == 4 or event.delta > 0:  # Scroll up
                self.zoom_factor *= 1.1
            elif event.num == 5 or event.delta < 0:  # Scroll down
                self.zoom_factor /= 1.1
                if self.zoom_factor < 0.1:
                    self.zoom_factor = 0.1
            
            self.resize_image_for_display()
    
    def enhance_image(self):
        if self.original_image is None:
            return
        
        try:
            # Convert to PIL Image for enhancement
            img = cv2.cvtColor(self.original_image, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            
            # Apply brightness adjustment
            brightness = self.image_enhancements["brightness"]
            if brightness != 0:
                from PIL import ImageEnhance
                enhancer = ImageEnhance.Brightness(pil_img)
                factor = 1.0 + (brightness / 100.0)
                pil_img = enhancer.enhance(factor)
            
            # Apply contrast adjustment
            contrast = self.image_enhancements["contrast"]
            if contrast != 1.0:
                from PIL import ImageEnhance
                enhancer = ImageEnhance.Contrast(pil_img)
                pil_img = enhancer.enhance(contrast)
            
            # Apply sharpness adjustment
            sharpness = self.image_enhancements["sharpness"]
            if sharpness != 1.0:
                from PIL import ImageEnhance
                enhancer = ImageEnhance.Sharpness(pil_img)
                pil_img = enhancer.enhance(sharpness)
            
            # Convert back to OpenCV format
            enhanced_img = np.array(pil_img)
            
            # Update displayed image
            self.displayed_image = enhanced_img
            
            # If we have a cropped image, apply the same enhancements
            if self.cropped_image is not None:
                # Convert cropped image to PIL
                cropped_rgb = cv2.cvtColor(self.cropped_image, cv2.COLOR_BGR2RGB)
                cropped_pil = Image.fromarray(cropped_rgb)
                
                # Apply same enhancements
                if brightness != 0:
                    enhancer = ImageEnhance.Brightness(cropped_pil)
                    factor = 1.0 + (brightness / 100.0)
                    cropped_pil = enhancer.enhance(factor)
                
                if contrast != 1.0:
                    enhancer = ImageEnhance.Contrast(cropped_pil)
                    cropped_pil = enhancer.enhance(contrast)
                
                if sharpness != 1.0:
                    enhancer = ImageEnhance.Sharpness(cropped_pil)
                    cropped_pil = enhancer.enhance(sharpness)
                
                # Convert back to OpenCV
                enhanced_cropped = np.array(cropped_pil)
                self.cropped_image = cv2.cvtColor(enhanced_cropped, cv2.COLOR_RGB2BGR)
            
            # Update display
            self.resize_image_for_display()
            self.status_var.set("Image enhanced")
            
        except Exception as e:
            self.status_var.set(f"Error enhancing image: {e}")
    
    def enable_crop_mode(self):
        if self.original_image is not None:
            self.is_cropping = True
            self.status_var.set("Crop mode enabled. Click and drag to select area.")
    
    def on_mouse_down(self, event):
        if self.is_cropping and self.original_image is not None:
            # Get canvas coordinates
            self.crop_start_x = self.canvas.canvasx(event.x)
            self.crop_start_y = self.canvas.canvasy(event.y)
            
            # Create rectangle if it doesn't exist
            if self.crop_rect:
                self.canvas.delete(self.crop_rect)
            
            self.crop_rect = self.canvas.create_rectangle(
                self.crop_start_x, self.crop_start_y, 
                self.crop_start_x, self.crop_start_y,
                outline="red", width=2
            )
    
    def on_mouse_drag(self, event):
        if self.is_cropping and self.crop_rect:
            # Get canvas coordinates
            x = self.canvas.canvasx(event.x)
            y = self.canvas.canvasy(event.y)
            
            # Update rectangle
            self.canvas.coords(
                self.crop_rect,
                self.crop_start_x, self.crop_start_y,
                x, y
            )
    
    def on_mouse_up(self, event):
        if self.is_cropping and self.crop_rect and self.original_image is not None:
            # Get canvas coordinates
            x = self.canvas.canvasx(event.x)
            y = self.canvas.canvasy(event.y)
            
            # Get coordinates
            x1, y1, x2, y2 = self.canvas.coords(self.crop_rect)
            
            # Ensure correct order (x1 < x2, y1 < y2)
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            
            # Convert canvas coordinates to original image coordinates
            # Account for zoom factor
            img_height, img_width = self.original_image.shape[:2]
            
            # Calculate scaling factors
            scale_factor = 1.0 / self.zoom_factor
            
            # Scale coordinates
            orig_x1 = int(x1 * scale_factor)
            orig_y1 = int(y1 * scale_factor)
            orig_x2 = int(x2 * scale_factor)
            orig_y2 = int(y2 * scale_factor)
            
            # Ensure coordinates are within image bounds
            orig_x1 = max(0, min(orig_x1, img_width - 1))
            orig_y1 = max(0, min(orig_y1, img_height - 1))
            orig_x2 = max(0, min(orig_x2, img_width - 1))
            orig_y2 = max(0, min(orig_y2, img_height - 1))
            
            # Crop the image
            self.cropped_image = self.original_image[orig_y1:orig_y2, orig_x1:orig_x2]
            
            # Display cropped image
            if self.cropped_image.size > 0:
                # Convert to RGB for display
                rgb_cropped = cv2.cvtColor(self.cropped_image, cv2.COLOR_BGR2RGB)
                self.displayed_image = rgb_cropped
                
                # Reset zoom for the cropped image
                self.zoom_factor = 1.0
                self.resize_image_for_display()
                self.status_var.set("Image cropped successfully")
            else:
                self.status_var.set("Invalid crop selection")
            
            # Disable cropping mode
            self.is_cropping = False
    
    def reset_crop(self):
        if self.original_image is not None:
            # Reset to original image
            self.displayed_image = cv2.cvtColor(self.original_image, cv2.COLOR_BGR2RGB)
            self.cropped_image = None
            
            # Reset zoom
            self.zoom_factor = 1.0
            self.resize_image_for_display()
            
            # Clear crop rectangle
            if self.crop_rect:
                self.canvas.delete(self.crop_rect)
                self.crop_rect = None
            
            self.is_cropping = False
            self.status_var.set("Crop reset")
    
    def preprocess_for_crnn(self, crop):
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (self.img_size[1], self.img_size[0]))  # Width, Height
        normalized = resized / 255.0
        expanded = np.expand_dims(normalized, axis=-1)
        return expanded
    
    def ctc_decoder(self, preds):
        try:
            decoded = tf.keras.backend.ctc_decode(
                preds, 
                input_length=np.ones(preds.shape[0]) * preds.shape[1]
            )[0][0]
            
            text = []
            for seq in decoded.numpy():
                line = ''.join([self.char_list[i] for i in seq if i != -1 and i < len(self.char_list)])
                text.append(line)
            return text
        except Exception as e:
            print(f"Error in CTC decoder: {e}")
            return ["Error decoding"]
    
    def predict_threaded(self):
        """Run prediction in a separate thread to keep UI responsive"""
        if not self.image_path:
            messagebox.showinfo("Error", "Please upload an image first")
            return
        
        if not self.dependency_check_complete:
            messagebox.showinfo("Please wait", "Still checking dependencies. Please try again in a moment.")
            return
        
        # Clear previous results
        self.result_text.delete(1.0, tk.END)
        self.status_var.set("Processing...")
        self.progress_var.set(0)
        self.root.update()
        
        # Start prediction in a separate thread
        thread = threading.Thread(target=self.predict)
        thread.daemon = True
        thread.start()
    
    def predict(self):
        try:
            # Check if PaddleOCR is available
            if self.ocr_detector is None:
                self.result_text.insert(tk.END, "PaddleOCR model not loaded. Please check dependencies.\n")
                self.status_var.set("Error: PaddleOCR not available")
                return
            
            # Use cropped image if available, otherwise use original
            img_to_process = self.cropped_image if self.cropped_image is not None else self.original_image
            
            # Create a temporary file for PaddleOCR
            temp_path = "temp_image.jpg"
            cv2.imwrite(temp_path, img_to_process)
            
            self.progress_var.set(10)
            self.root.update()
            
            # Detect text boxes with PaddleOCR
            self.result_text.insert(tk.END, "Running PaddleOCR detection...\n")
            result = self.ocr_detector.ocr(temp_path, cls=True)
            
            self.progress_var.set(50)
            self.root.update()
            
            if result[0] is None:
                self.result_text.insert(tk.END, "No text detected in the image.\n")
                self.status_var.set("Prediction complete - No text detected")
                self.progress_var.set(100)
                return
                
            boxes = result[0]
            
            # Get confidence threshold
            threshold = self.threshold_var.get()
            
            # Create a copy for visualization
            vis_image = img_to_process.copy()
            
            # Process each detected box
            all_predictions = []
            filtered_boxes = []
            
            self.result_text.insert(tk.END, f"Found {len(boxes)} text regions\n")
            self.result_text.insert(tk.END, f"Using confidence threshold: {threshold:.2f}\n\n")
            
            for idx, box in enumerate(boxes):
                # Update progress
                progress = 50 + (idx / len(boxes)) * 40
                self.progress_var.set(progress)
                self.root.update()
                
                coords = np.array(box[0]).astype(int)
                paddle_text = box[1][0]  # PaddleOCR's text prediction
                confidence = box[1][1]   # PaddleOCR's confidence
                
                # Skip if below threshold
                if confidence < threshold:
                    continue
                
                filtered_boxes.append(box)
                
                x1, y1 = coords[0]
                x2, y2 = coords[2]
                
                # Crop region
                cropped = img_to_process[y1:y2, x1:x2]
                if cropped.size == 0:
                    continue  # Skip invalid boxes
                
                # CRNN prediction if model is loaded
                crnn_text = ""
                if self.model_loaded and cropped.size > 0:
                    input_img = self.preprocess_for_crnn(cropped)
                    input_img = np.expand_dims(input_img, axis=0)  # Add batch dimension
                    
                    preds = self.crnn_model.predict(input_img, verbose=0)
                    crnn_text = self.ctc_decoder(preds)[0]
                
                # Combine results
                result_text = f"Box {idx+1}:\n"
                result_text += f"  PaddleOCR: {paddle_text} (Conf: {confidence:.2f})\n"
                if crnn_text:
                    result_text += f"  CRNN: {crnn_text}\n"
                result_text += f"  Location: ({x1},{y1}) to ({x2},{y2})\n\n"
                
                all_predictions.append(result_text)
                
                # Draw on visualization image
                color = (0, int(255 * confidence), int(255 * (1 - confidence)))  # Color based on confidence
                cv2.rectangle(vis_image, (x1, y1), (x2, y2), color, 2)
                cv2.putText(vis_image, f"{idx+1}: {confidence:.2f}", (x1, y1-5), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            
            self.progress_var.set(90)
            self.root.update()
            
            # Display results
            for pred in all_predictions:
                self.result_text.insert(tk.END, pred)
            
            # Save visualization
            cv2.imwrite("output_with_predictions.jpg", vis_image)
            
            # Update displayed image with visualization
            rgb_vis = cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB)
            self.displayed_image = rgb_vis
            self.resize_image_for_display()
            
            # Clean up
            if os.path.exists(temp_path):
                os.remove(temp_path)
                
            self.status_var.set(f"Prediction complete - {len(filtered_boxes)} text regions found")
            self.progress_var.set(100)
            
        except Exception as e:
            self.result_text.insert(tk.END, f"Error during prediction: {str(e)}\n")
            self.result_text.insert(tk.END, "Try installing dependencies manually:\n")
            self.result_text.insert(tk.END, "pip install paddlepaddle paddleocr\n")
            self.status_var.set("Error during prediction")
            self.progress_var.set(0)
            print(f"Error: {e}")
    
    def save_results(self):
        """Save the detected text to a file"""
        if not self.result_text.get(1.0, tk.END).strip():
            messagebox.showinfo("Error", "No results to save")
            return
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
        )
        
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(self.result_text.get(1.0, tk.END))
                self.status_var.set(f"Results saved to {os.path.basename(file_path)}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to save file: {e}")

if __name__ == "__main__":
    # Set up better exception handling
    def show_error(exc_type, exc_value, exc_traceback):
        import traceback
        error_msg = ''.join(traceback.format_exception(exc_type, exc_value, exc_traceback))
        messagebox.showerror("Error", f"An unexpected error occurred:\n\n{error_msg}")
        print(error_msg)
    
    # Set up the exception hook
    sys.excepthook = show_error
    
    # Create and run the application
    root = tk.Tk()
    app = PrescriptionOCRApp(root)
    
    # Configure window resize behavior
    def on_resize(event):
        if event.widget == root:
            app.resize_image_for_display()
    
    root.bind("<Configure>", on_resize)
    
    # Center window on screen
    window_width = 1200
    window_height = 800
    screen_width = root.winfo_screenwidth()
    screen_height = root.winfo_screenheight()
    center_x = int(screen_width/2 - window_width/2)
    center_y = int(screen_height/2 - window_height/2)
    root.geometry(f'{window_width}x{window_height}+{center_x}+{center_y}')
    
    root.mainloop()

C:\Users\vaibh\miniconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [5]:
pip install paddlepaddle

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install paddleocr

In [3]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en')  # load model

img_path = 'path_to_image.jpg'

result = ocr.ocr(img_path, cls=True)

print(result)


C:\Users\vaibh\miniconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


OSError: [WinError 127] The specified procedure could not be found. Error loading "C:\Users\vaibh\miniconda3\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [2]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Found existing installation: torch 2.4.1
Uninstalling torch-2.4.1:
  Successfully uninstalled torch-2.4.1
Found existing installation: torchvision 0.19.1
Uninstalling torchvision-0.19.1:
  Successfully uninstalled torchvision-0.19.1
Found existing installation: torchaudio 2.4.1
Uninstalling torchaudio-2.4.1:
  Successfully uninstalled torchaudio-2.4.1
Files removed: 1540
Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/215.2 MB ? eta -:--:--
   ---------------------------------------- 0.2/215.2 MB 4.8 MB/s eta 0:00:45
   ---------------------------------------- 0.6/215.2 MB 6.4 MB/s eta 0:00:34
   ---------------------------------------- 0.7/215.2 MB 6.4 MB/s eta 0:00:34
   ---------------------------------------- 0.7/215.2 MB 6.4 MB/s eta 0:00:34
   ---------------------------------------- 0.9/215.2 MB 4.2 MB/s eta 0:00:51
   ---------------------------------------- 0.9/215.2 MB 4.2 MB/s eta 0:00:51
   ---------------------------

In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

OSError: [WinError 127] The specified procedure could not be found. Error loading "C:\Users\vaibh\miniconda3\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.